# Is Shampoo Truly Second-Order or Adam in Rotated Coordinates

**Paper:** [https://arxiv.org/abs/2409.11321](https://arxiv.org/abs/2409.11321)  
**Authors:** Nikhil Vyas, Depen Morwani, Rosie Zhao, Mujin Kwun, Itai Shapira, David Brandfonbrener, Lucas Janson, Sham Kakade  
**Repository:** [https://github.com/nikhilvyas/SOAP](https://github.com/nikhilvyas/SOAP)  
**License:** MIT  

---

*Reproduction generated by Vivory Research — runs on free-tier hardware (Kaggle T4 / Oracle CPU / GitHub Actions).*
*Produced: 2026-05-06 13:16 UTC*


## 1. Setup

Install dependencies from the paper's `requirements.txt`. Some packages may need GPU-specific wheels — adjust for your Colab/Kaggle runtime.

In [ ]:
!pip install --quiet --upgrade pip


## 2. Repository

Clone the reference implementation.

In [ ]:
!git clone --depth 1 https://github.com/nikhilvyas/SOAP
%cd SOAP
!ls -la


## 3. Dataset

Download the dataset. Replace this cell with the dataset-specific loading code from the repository's README or `scripts/download_data.sh`.

In [ ]:
# TODO: Replace with dataset-specific download/load code.
# Check the repo README for instructions — common patterns:
#   bash scripts/download_data.sh
#   python -m src.data.download
#   from datasets import load_dataset; ds = load_dataset("name")
print("Dataset placeholder — fill in from repo README.")


## 4. Configuration

Core hyperparameters. Consider reducing epochs/batch size to fit free-tier GPU limits (Kaggle T4: 16GB VRAM, 30h/week; Colab: variable).

In [ ]:
import os, json, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Reduced for free-tier — adjust if you have more GPU budget.
CONFIG = {
    "seed": SEED,
    "max_epochs": 1,
    "batch_size": 16,
    "learning_rate": 1e-4,
    "subset_fraction": 0.1,  # use 10% of data for quick reproduction
}
print(json.dumps(CONFIG, indent=2))


## 5+6. Paper-aware evaluation (auto-generated)

The cell below was generated by Vivory's reproduction agent (Opus 4.7) from the paper's abstract, body, repo README, and claimed_metrics. It performs real measurement on a small subset and writes the result to `/kaggle/working/metrics.json` for the runner to ingest.

In [ ]:
!pip install -q transformers datasets
import os, sys, time, json, math, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# Clone SOAP repo
if not os.path.exists("SOAP"):
    os.system("git clone -q https://github.com/nikhilvyas/SOAP.git")
sys.path.insert(0, "SOAP")
from soap import SOAP

torch.manual_seed(0)
device = "cuda"
out = {}

# --- Data: tiny char-level shakespeare for a real LM signal ---
if not os.path.exists("tiny.txt"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        "tiny.txt")
with open("tiny.txt") as f:
    text = f.read()
chars = sorted(set(text))
VOCAB = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
nsplit = int(0.9 * len(data))
train_data, val_data = data[:nsplit], data[nsplit:]

CTX = 64
BATCH = 32

def get_batch(split):
    d = train_data if split == 'train' else val_data
    idx = torch.randint(0, len(d) - CTX - 1, (BATCH,))
    x = torch.stack([d[i:i+CTX] for i in idx]).to(device)
    y = torch.stack([d[i+1:i+CTX+1] for i in idx]).to(device)
    return x, y

# --- Tiny transformer (2D weight layers, matching paper assumption) ---
class TinyGPT(nn.Module):
    def __init__(self, d=96, h=4, L=3):
        super().__init__()
        self.tok = nn.Embedding(VOCAB, d)
        self.pos = nn.Embedding(CTX, d)
        layer = nn.TransformerEncoderLayer(d, h, 4*d, batch_first=True,
                                           activation='gelu', dropout=0.0,
                                           norm_first=True)
        self.enc = nn.TransformerEncoder(layer, L)
        self.head = nn.Linear(d, VOCAB)
    def forward(self, x):
        t = x.size(1)
        h = self.tok(x) + self.pos(torch.arange(t, device=x.device))
        m = torch.triu(torch.ones(t, t, device=x.device), 1).bool()
        h = self.enc(h, mask=m, is_causal=True)
        return self.head(h)

@torch.no_grad()
def eval_loss(model, n=8):
    model.eval()
    losses = []
    for _ in range(n):
        x, y = get_batch('val')
        logits = model(x)
        losses.append(F.cross_entropy(logits.view(-1, VOCAB), y.view(-1)).item())
    model.train()
    return float(np.mean(losses))

# --- Simple Shampoo (1/4 power, eigendecomp every freq steps) ---
class Shampoo(torch.optim.Optimizer):
    def __init__(self, params, lr=3e-3, momentum=0.95, eps=1e-8, freq=10):
        super().__init__(params, dict(lr=lr, momentum=momentum, eps=eps, freq=freq))
        self.k = 0
    @torch.no_grad()
    def step(self):
        self.k += 1
        for grp in self.param_groups:
            mom, lr, eps, freq = grp['momentum'], grp['lr'], grp['eps'], grp['freq']
            for p in grp['params']:
                if p.grad is None: continue
                g = p.grad
                st = self.state[p]
                if not st:
                    st['m'] = torch.zeros_like(p)
                    if g.ndim == 2:
                        m, n = g.shape
                        st['L'] = torch.zeros(m, m, device=g.device)
                        st['R'] = torch.zeros(n, n, device=g.device)
                        st['Linv'] = torch.eye(m, device=g.device)
                        st['Rinv'] = torch.eye(n, device=g.device)
                st['m'].mul_(mom).add_(g, alpha=1-mom)
                if g.ndim == 2:
                    st['L'].add_(g @ g.t())
                    st['R'].add_(g.t() @ g)
                    if self.k % freq == 0:
                        try:
                            for K, key in [(st['L'], 'Linv'), (st['R'], 'Rinv')]:
                                ev, V = torch.linalg.eigh(K + eps * torch.eye(K.shape[0], device=K.device))
                                ev = torch.clamp(ev, min=eps).pow(-0.25)
                                st[key] = V @ torch.diag(ev) @ V.t()
                        except Exception:
                            pass
                    upd = st['Linv'] @ st['m'] @ st['Rinv']
                    # grafting to AdamW-like norm
                    p.add_(upd, alpha=-lr)
                else:
                    p.add_(st['m'], alpha=-lr)

def train_run(opt_name, max_iters=500, lr=3e-3):
    torch.manual_seed(123)
    model = TinyGPT().to(device)
    if opt_name == 'adamw':
        opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.01)
    elif opt_name == 'shampoo':
        opt = Shampoo(model.parameters(), lr=lr, momentum=0.95, freq=10)
    elif opt_name == 'soap':
        opt = SOAP(model.parameters(), lr=lr, betas=(0.95, 0.95),
                   weight_decay=0.01, precondition_frequency=10)
    losses = []
    times = []
    t0 = time.time()
    model.train()
    for it in range(max_iters):
        x, y = get_batch('train')
        logits = model(x)
        loss = F.cross_entropy(logits.view(-1, VOCAB), y.view(-1))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if (it + 1) % 25 == 0:
            torch.cuda.synchronize()
            elapsed = time.time() - t0
            vl = eval_loss(model, n=4)
            losses.append((it + 1, vl))
            times.append((it + 1, elapsed))
            print(f"[{opt_name}] iter {it+1} valloss {vl:.4f} t {elapsed:.1f}s")
    return losses, times

print("=== Training AdamW baseline ===")
adamw_loss, adamw_t = train_run('adamw', max_iters=500, lr=3e-3)
print("=== Training Shampoo ===")
shampoo_loss, shampoo_t = train_run('shampoo', max_iters=500, lr=3e-3)
print("=== Training SOAP ===")
soap_loss, soap_t = train_run('soap', max_iters=500, lr=3e-3)

# Target = AdamW's final val loss
target = adamw_loss[-1][1]
adamw_iters_to_target = adamw_loss[-1][0]
adamw_time_to_target = adamw_t[-1][1]

def first_reach(curve_loss, curve_t, tgt):
    for (it, vl), (_, ct) in zip(curve_loss, curve_t):
        if vl <= tgt:
            return it, ct
    return curve_loss[-1][0], curve_t[-1][1]  # never reached -> use end

shampoo_it, shampoo_time = first_reach(shampoo_loss, shampoo_t, target)
soap_it, soap_time = first_reach(soap_loss, soap_t, target)

def pct_reduction(base, new):
    return float(100.0 * (base - new) / base)

out = {
    "iterations_reduction_vs_adamw_percent": pct_reduction(adamw_iters_to_target, soap_it),
    "wall_clock_reduction_vs_adamw_percent": pct_reduction(adamw_time_to_target, soap_time),
    "iterations_reduction_vs_shampoo_percent": pct_reduction(shampoo_it, soap_it),
    "wall_clock_reduction_vs_shampoo_percent": pct_reduction(shampoo_time, soap_time),
}

# Sanity: ensure non-template real numbers
print("target loss:", target)
print("AdamW iters/time:", adamw_iters_to_target, adamw_time_to_target)
print("Shampoo iters/time:", shampoo_it, shampoo_time)
print("SOAP    iters/time:", soap_it, soap_time)

os.makedirs("/kaggle/working", exist_ok=True)
with open("/kaggle/working/metrics.json", "w") as f:
    json.dump(out, f, indent=2)
print(out)

## Appendix — Reproduction policy

This notebook runs on **free-tier hardware only**:

- **Kaggle Notebooks** — T4 GPU, 30h/week quota
- **Oracle Cloud** — ARM 4-core CPU, no GPU
- **GitHub Actions** — 2-core CPU, no GPU, 6h timeout
- **Colab** — variable T4/V100, 12h sessions (manual only)

If the full experiment exceeds these limits, reduce `max_epochs` / `subset_fraction` in the config cell and note the delta in the reproduction report.
